# Audio Psychoacoustic Compression: MDCT, Bark Scale, and Perceptual Masking

This notebook demonstrates the fundamental principles of perceptual audio compression as used in modern codecs like MP3, AAC, and Vorbis. We'll implement a simplified version of the key components:

1. **Modified Discrete Cosine Transform (MDCT)** - Time-frequency analysis
2. **Bark Scale Subbanding** - Perceptually meaningful frequency grouping
3. **Psychoacoustic Masking Model** - Human auditory perception modeling
4. **Perceptual Quantization** - Noise shaping below audibility threshold

## References

This implementation is based on the following key references:

1. **Princen, J. P., & Bradley, A. B. (1986)**. "Analysis/synthesis filter bank design based on time domain aliasing cancellation." *IEEE Transactions on Acoustics, Speech, and Signal Processing*, 34(5), 1153-1161.

2. **Brandenburg, K., & Stoll, G. (1994)**. "ISO-MPEG-1 audio: A generic standard for coding of high-quality digital audio." *Journal of the Audio Engineering Society*, 42(10), 780-792.

3. **Zwicker, E., & Fastl, H. (2013)**. *Psychoacoustics: Facts and models*. Springer Science & Business Media.

4. **Traunmüller, H. (1990)**. "Analytical expressions for the tonotopic sensory scale." *Journal of the Acoustical Society of America*, 88(1), 97-100.

5. **Johnston, J. D. (1988)**. "Transform coding of audio signals using perceptual noise criteria." *IEEE Journal on Selected Areas in Communications*, 6(2), 314-323.

## Background Theory

### MDCT (Modified Discrete Cosine Transform)

The MDCT is a lapped transform that provides:
- **Time-domain aliasing cancellation** when used with 50% overlap
- **Critical sampling** (N samples → N/2 coefficients)
- **Perfect reconstruction** with appropriate windowing

The forward MDCT is defined as:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot w[n] \cdot \cos\left(\frac{\2pi}{N}(n + n_0)(k + 0.5)\right)$$

where $n_0 = \frac{\frac{N}{2} + 1}{2}$

### Bark Scale

The Bark scale approximates the critical band structure of human hearing:
- **24 critical bands** from 20 Hz to 20 kHz
- **Non-linear frequency mapping** reflecting cochlear mechanics
- **Masking occurs primarily within critical bands**

### Psychoacoustic Masking

Human auditory masking involves:
- **Absolute Threshold of Hearing (ATH)** - minimum audible levels
- **Simultaneous masking** - strong signals mask weaker ones
- **Frequency spreading** - masking effect spreads across frequency

---

## Implementation

Let's start with the necessary imports:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from math import pi
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib style for better plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 1. MDCT Implementation

The MDCT provides perfect reconstruction when used with a sine window and 50% overlap. This satisfies the Princen-Bradley condition for time-domain aliasing cancellation.

In [ ]:
def sine_window(N):
    """
    Generate a sine window for MDCT analysis/synthesis.
    
    The sine window satisfies the Princen-Bradley condition:
    w²[n] + w²[n + N/2] = 1 for perfect reconstruction
    
    Parameters:
    -----------
    N : int
        Window length
        
    Returns:
    --------
    w : ndarray
        Sine window of length N
    """
    n = np.arange(N)
    return np.sin(pi/N * (n + 0.5))

def mdct(x, N):
    """
    Modified Discrete Cosine Transform with sine window and 50% overlap.
    
    This provides critical sampling: N input samples → N/2 coefficients
    
    Parameters:
    -----------
    x : ndarray
        Input time-domain signal
    N : int
        MDCT window size (transform length)
        
    Returns:
    --------
    X_frames : ndarray
        MDCT coefficients, shape (num_frames, N//2)
    hop : int
        Hop size (N//2 for 50% overlap)
    """
    w = sine_window(N)
    hop = N // 2
    # pad for full frames
    pad = (N - (len(x) % hop)) % hop
    x = np.concatenate([np.zeros(hop), x, np.zeros(hop + pad)])
    frames = []
    
    # Calculate n₀ = (N/2 + 1)/2 as per the reference image
    n0 = (N/2 + 1) / 2
    
    for i in range(0, len(x) - N + 1, hop):
        xw = x[i:i+N] * w
        # MDCT transform matrix as per reference image
        n = np.arange(N)
        k = np.arange(N//2)[:, None]
        # MDCT formula: cos(2π/N * (n + n₀) * (k + 1/2))
        C = np.cos((2*pi/N) * (n + n0) * (k + 0.5))
        X = np.dot(C, xw)
        frames.append(X)
    return np.array(frames), hop

def imdct(X, N):
    """
    Inverse Modified Discrete Cosine Transform with overlap-add reconstruction.
    
    Perfect reconstruction is achieved through:
    1. Inverse transform using C^T
    2. Proper scaling (2/N)
    3. Windowing and overlap-add
    
    Parameters:
    -----------
    X : ndarray
        MDCT coefficients, shape (num_frames, N//2)
    N : int
        MDCT window size
        
    Returns:
    --------
    y : ndarray
        Reconstructed time-domain signal
    """
    w = sine_window(N)
    hop = N // 2
    num_frames = X.shape[0]
    out_len = num_frames * hop + hop + hop  # matches mdct padding
    y = np.zeros(out_len)
    
    # Calculate n₀ = (N/2 + 1)/2 as per the reference image
    n0 = (N/2 + 1) / 2
    
    n = np.arange(N)
    k = np.arange(N//2)[:, None]
    # IMDCT formula as per reference image: cos(2π/N * (n + n₀) * (k + 1/2))
    C = np.cos((2*pi/N) * (n + n0) * (k + 0.5))
    for i in range(num_frames):
        xw = np.dot(C.T, X[i])  # inverse transform
        # Scale by 4/N as per reference image
        xw = (4.0/N) * xw
        # Apply synthesis window for overlap-add reconstruction
        xw = xw * w
        start = i * hop
        y[start:start+N] += xw
    # Remove the analysis pre/post padding used in mdct()
    return y[hop:-hop]

## 2. Bark Scale and Critical Band Analysis

The Bark scale divides the audible frequency range into 24 critical bands that correspond to the frequency resolution of the human auditory system.

In [ ]:
def hz_to_bark(freq_hz):
    """
    Convert frequency in Hz to Bark scale using Traunmüller's approximation (1990).
    
    The Bark scale approximates the critical band rate of human hearing.
    There are approximately 24 Bark bands from 20 Hz to 20 kHz.
    
    Parameters:
    -----------
    freq_hz : float or ndarray
        Frequency in Hz
        
    Returns:
    --------
    bark : float or ndarray
        Frequency in Bark scale
        
    Reference:
    ----------
    Traunmüller, H. (1990). "Analytical expressions for the tonotopic 
    sensory scale." JASA, 88(1), 97-100.
    """
    return 26.81 / (1 + 1960.0 / freq_hz) - 0.53

def bark_band_edges(fs, N):
    """
    Map MDCT bins to Bark-scale bands.
    
    MDCT bins correspond to frequencies centered at (k+0.5)*fs/(2N)
    where k is the bin index (0 to N/2-1).
    
    Parameters:
    -----------
    fs : int
        Sampling rate in Hz
    N : int
        MDCT transform size
        
    Returns:
    --------
    freqs : ndarray
        Center frequencies of MDCT bins
    bark : ndarray
        Bark values for each bin
    band_idx : ndarray
        Bark band index (0-24) for each MDCT bin
    band_centers : ndarray
        Center frequency of each Bark band
    """
    k = np.arange(N//2)
    freqs = (k + 0.5) * fs / (2*N)
    bark = hz_to_bark(np.maximum(freqs, 1.0))  # Avoid division by zero
    
    # Map to integer Bark bands (0 to 24)
    band_idx = np.clip(np.floor(bark).astype(int), 0, 24)
    
    # Compute center frequency for each Bark band
    band_centers = np.array([
        np.mean(freqs[band_idx == b]) if np.any(band_idx == b) else 0 
        for b in range(25)
    ])
    
    return freqs, bark, band_idx, band_centers

## 3. Psychoacoustic Masking Model

The psychoacoustic model estimates the masking threshold - the level below which quantization noise becomes inaudible. This combines:

1. **Absolute Threshold of Hearing (ATH)** - the minimum audible threshold in quiet
2. **Simultaneous masking** - how strong signals mask weaker ones
3. **Frequency spreading** - masking effects spread across frequency according to the spreading function

In [ ]:
def absolute_threshold_of_hearing(freq_hz):
    """
    Approximate Absolute Threshold of Hearing as a function of frequency.
    
    This represents the minimum sound pressure level audible to the average
    human ear in a quiet environment. The curve shows high sensitivity 
    around 2-5 kHz and decreased sensitivity at very low and high frequencies.
    
    Parameters:
    -----------
    freq_hz : ndarray
        Frequencies in Hz
        
    Returns:
    --------
    ath : ndarray
        Threshold levels in dB SPL
        
    Reference:
    ----------
    Based on Zwicker & Fastl (2013), "Psychoacoustics: Facts and models"
    """
    f = np.maximum(freq_hz, 1.0) / 1000.0  # Frequency in kHz
    
    # Zwicker-like approximation
    ath = 3.64*(f**-0.8) - 6.5*np.exp(-0.6*(f-3.3)**2) + 1e-3*(f**4)
    
    return ath

def spreading_function_db(delta_bark):
    """
    Asymmetric spreading function in the Bark domain.
    
    Masking spreads asymmetrically in frequency:
    - Downward spread (low freq masking high freq): -24 dB/Bark
    - Upward spread (high freq masking low freq): -27 dB/Bark
    
    Parameters:
    -----------
    delta_bark : ndarray
        Bark distance from masker (negative = below masker)
        
    Returns:
    --------
    spread : ndarray
        Spreading attenuation in dB
        
    Reference:
    ----------
    Johnston, J. D. (1988). "Transform coding of audio signals using 
    perceptual noise criteria." IEEE JSAC, 6(2), 314-323.
    """
    down = -24  # dB/Bark downward spread
    up = -27    # dB/Bark upward spread
    
    return np.where(delta_bark < 0, down * (-delta_bark), up * delta_bark)

def estimate_masking_threshold_db(band_energies_db, band_centers_hz):
    """
    Estimate psychoacoustic masking threshold combining ATH and frequency spreading.
    
    For each time frame and frequency band, the masking threshold is computed as:
    1. Convert all masker energies to linear scale
    2. Apply spreading function to distribute masking across frequency
    3. Sum masking contributions from all bands
    4. Take maximum of summed masking and ATH
    
    Parameters:
    -----------
    band_energies_db : ndarray
        Signal energy per band per frame, shape (num_frames, num_bands)
    band_centers_hz : ndarray
        Center frequency of each band in Hz
        
    Returns:
    --------
    thresholds_db : ndarray
        Masking threshold per band per frame in dB
    """
    num_frames, num_bands = band_energies_db.shape
    
    # Convert frequencies to Bark scale
    bark_pos = hz_to_bark(np.maximum(band_centers_hz, 1.0))
    bark_pos[np.isnan(bark_pos)] = 0.0
    
    # Precompute spreading matrix (from band i to band j)
    delta = bark_pos[None, :] - bark_pos[:, None]  # Shape: (num_bands, num_bands)
    spread_db = spreading_function_db(delta)
    
    # Compute ATH for each band
    ath_db = absolute_threshold_of_hearing(np.maximum(band_centers_hz, 1.0))
    ath_db = np.where(np.isfinite(ath_db), ath_db, 
                      np.max(ath_db[np.isfinite(ath_db)]))
    
    thresholds_db = np.empty_like(band_energies_db)
    
    for t in range(num_frames):
        # Apply spreading: masker energy + spreading attenuation
        influence = band_energies_db[t][:, None] + spread_db
        
        # Convert to linear scale, sum contributions, back to dB
        inf_lin = 10**(influence/10.0)
        summed = np.sum(inf_lin, axis=0) + 1e-12
        
        # Combine masking and ATH (take maximum in linear domain)
        ath_lin = 10**(ath_db/10.0)
        max_lin = np.maximum(summed, ath_lin)
        thresholds_db[t] = 10*np.log10(max_lin + 1e-12)
    
    return thresholds_db

## 4. Perceptual Quantization

The quantization process shapes the noise spectrum to stay below the masking threshold. For each frequency band, we:

1. Calculate the masking threshold
2. Determine the allowable quantization noise level
3. Set the quantization step size accordingly
4. Apply uniform scalar quantization

In [ ]:
def quantize_mdct_per_band(X_frames, band_idx, thresholds_db, safety_db=6.0):
    """
    Perceptually-guided quantization of MDCT coefficients.
    
    For each Bark band, the quantization step is chosen so that the
    resulting quantization noise power remains below the masking threshold
    with a safety margin.
    
    Quantization noise model:
    - For uniform quantization with step q: noise_variance = q²/12
    - Total noise in band = noise_variance × number_of_bins
    
    Parameters:
    -----------
    X_frames : ndarray
        MDCT coefficients, shape (num_frames, num_bins)
    band_idx : ndarray
        Bark band index for each MDCT bin
    thresholds_db : ndarray
        Masking thresholds per band per frame in dB
    safety_db : float
        Safety margin below masking threshold in dB
        
    Returns:
    --------
    Xq : ndarray
        Quantized (and dequantized) MDCT coefficients
    """
    num_frames, num_bins = X_frames.shape
    nbands = thresholds_db.shape[1]
    Xq = np.zeros_like(X_frames)
    
    # Precompute band masks for efficiency
    band_masks = [np.where(band_idx == b)[0] for b in range(nbands)]
    
    for t in range(num_frames):
        for b in range(nbands):
            bins = band_masks[b]
            if len(bins) == 0:
                continue
                
            coeffs = X_frames[t, bins]
            
            # Masking threshold for this band (convert dB to linear power)
            mask_thresh_db = thresholds_db[t, b]
            mask_thresh_pow = 10**(mask_thresh_db/10.0)
            
            # Target noise power (below threshold with safety margin)
            target_noise_pow = mask_thresh_pow / (10**(safety_db/10.0))
            
            # Uniform quantization noise model: q²/12 per coefficient
            # Total noise in band = (q²/12) × num_bins
            num_bins_in_band = len(bins)
            noise_var_per_bin = target_noise_pow / num_bins_in_band
            q = np.sqrt(12.0 * noise_var_per_bin)
            
            # Ensure reasonable quantization step
            if not np.isfinite(q) or q <= 0:
                q = 1e-6
            
            # Prevent over-quantization that destroys the signal
            max_coeff = np.max(np.abs(coeffs))
            if max_coeff > 0 and q > max_coeff / 2.0:
                q = max_coeff / 2.0
            
            # Apply uniform scalar quantization
            Xi = np.round(coeffs / q)
            Xq[t, bins] = Xi * q
            #Xq[t,bins] = coeffs
    
    return Xq

## 5. Test Signal Generation

We'll create a test signal that demonstrates the masking effects:
- Multiple tones at different frequencies
- Pink noise (1/f spectrum) background

In [ ]:
# Audio parameters
fs = 48000  # Sampling rate
dur = 2.0   # Duration in seconds
t = np.arange(int(fs*dur)) / fs

# Create test signal: three tones at different frequencies
sig = (
    0.5*np.sin(2*pi*440*t) +     # A4 (440 Hz)
    0.3*np.sin(2*pi*1000*t) +    # 1 kHz tone
    0.2*np.sin(2*pi*3500*t)      # 3.5 kHz tone
)

# Add pink noise (1/f spectrum)
rng = np.random.default_rng(42)  # Reproducible random seed
white = rng.standard_normal(sig.shape[0])

# Shape noise spectrum to 1/f (pink noise)
F = np.fft.rfft(white)
freqs = np.fft.rfftfreq(len(white), 1/fs)
shape = 1/np.maximum(freqs, 1.0)  # 1/f shaping
pink = np.fft.irfft(F * shape / np.max(shape))
pink = pink / np.max(np.abs(pink)) * 0.1  # Scale to 10% of signal level

# Combine signal and noise
x = sig + pink
x = x / np.max(np.abs(x)) * 0.9  # Normalize to prevent clipping

print(f"Test signal: {len(x)} samples, {dur} seconds at {fs} Hz")
print(f"Signal contains: 440 Hz, 1 kHz, 3.5 kHz tones + pink noise")

## 6. MDCT Analysis

Transform the audio signal into the MDCT domain for frequency analysis.

In [ ]:
# MDCT parameters
N = 1024  # Window size (typical for audio compression)

# Perform MDCT analysis
X_frames, hop = mdct(x, N)
T_frames, N2 = X_frames.shape

print(f"MDCT Analysis:")
print(f"  Window size: {N} samples")
print(f"  Hop size: {hop} samples (50% overlap)")
print(f"  Frames: {T_frames}")
print(f"  Coefficients per frame: {N2}")
print(f"  Frequency resolution: {fs/(2*N):.2f} Hz per bin")

## 7. Bark Scale Subbanding

Group MDCT bins into perceptually meaningful Bark-scale subbands.

In [ ]:
# Map MDCT bins to Bark bands
freqs, bark_vals, band_idx, band_centers = bark_band_edges(fs, N)
nbands = 25  # Standard number of Bark bands

# Calculate energy per band per frame
band_energy = np.zeros((T_frames, nbands))
for b in range(nbands):
    bins = np.where(band_idx == b)[0]
    if len(bins) == 0:
        continue
    # Mean energy per bin in the band
    band_energy[:, b] = np.mean(X_frames[:, bins]**2, axis=1) + 1e-20

# Convert to dB scale
band_energy_db = 10*np.log10(band_energy)

print(f"Bark Scale Analysis:")
print(f"  Number of Bark bands: {nbands}")
print(f"  Frequency range: {freqs[0]:.1f} - {freqs[-1]:.1f} Hz")
print(f"  Bark range: {bark_vals[0]:.2f} - {bark_vals[-1]:.2f}")

# Show band allocation
bands_with_content = np.unique(band_idx)
print(f"  Active bands: {len(bands_with_content)} of {nbands}")

## 8. Psychoacoustic Masking Analysis

Estimate the masking threshold for each band and time frame.

In [ ]:
# Estimate masking thresholds
thresholds_db = estimate_masking_threshold_db(band_energy_db, band_centers)

print(f"Psychoacoustic Analysis:")
print(f"  Masking thresholds shape: {thresholds_db.shape}")
print(f"  Threshold range: {np.min(thresholds_db):.1f} to {np.max(thresholds_db):.1f} dB")

# Calculate average masking across time
avg_band_energy_db = np.mean(band_energy_db, axis=0)
avg_threshold_db = np.mean(thresholds_db, axis=0)

# Show masking effectiveness
masking_benefit_db = avg_band_energy_db - avg_threshold_db
print(f"  Average masking benefit: {np.mean(masking_benefit_db[masking_benefit_db > 0]):.1f} dB")

## 9. Perceptual Quantization and Reconstruction

Apply psychoacoustically-guided quantization and reconstruct the audio.

In [ ]:
# Apply perceptual quantization
safety_margin_db = 6.0  # Safety margin below masking threshold
Xq_frames = quantize_mdct_per_band(X_frames, band_idx, thresholds_db, safety_margin_db)

# Reconstruct audio via IMDCT
y = imdct(Xq_frames, N)

print(f"Quantization and Reconstruction:")
print(f"  Safety margin: {safety_margin_db} dB")
print(f"  Original length: {len(x)} samples")
print(f"  Reconstructed length: {len(y)} samples")

# Calculate compression metrics
# Ensure same length for comparison
min_len = min(len(x), len(y))
x_comp = x[:min_len]
y_comp = y[:min_len]

# Calculate SNR and other metrics
mse = np.mean((x_comp - y_comp)**2)
signal_power = np.mean(x_comp**2)
noise_power = np.mean((x_comp - y_comp)**2)
snr_db = 10*np.log10(signal_power / (noise_power + 1e-12))

print(f"\nQuality Metrics:")
print(f"  MSE: {mse:.6f}")
print(f"  SNR: {snr_db:.2f} dB")
print(f"  Max absolute error: {np.max(np.abs(x_comp - y_comp)):.6f}")

## 10. Visualization and Analysis

Let's visualize the results to understand how the psychoacoustic compression works.

In [ ]:
# 1. Waveform comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Time segment for detailed view (first 50ms)
seg_samples = int(0.05 * fs)
t_seg = t[:seg_samples]

ax1.plot(t_seg*1000, x_comp[:seg_samples], 'b-', label='Original', linewidth=1)
ax1.plot(t_seg*1000, y_comp[:seg_samples], 'r--', label='Reconstructed', alpha=0.8, linewidth=1)
ax1.set_title('Waveform Comparison (First 50ms)', fontsize=14)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Amplitude')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Error signal
error = x_comp - y_comp
ax2.plot(t[:min_len]*1000, error, 'g-', linewidth=0.5, alpha=0.7)
ax2.set_title(f'Reconstruction Error (SNR = {snr_db:.1f} dB)', fontsize=14)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Error Amplitude')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 2. Bark band energy vs masking threshold
plt.figure(figsize=(14, 8))

# Only show bands with content
active_bands = bands_with_content
plt.plot(active_bands, avg_band_energy_db[active_bands], 'bo-', 
         label='Average Band Energy', linewidth=2, markersize=6)
plt.plot(active_bands, avg_threshold_db[active_bands], 'rs-', 
         label='Average Masking Threshold', linewidth=2, markersize=6)

plt.fill_between(active_bands, avg_threshold_db[active_bands], 
                 avg_band_energy_db[active_bands], 
                 where=(avg_band_energy_db[active_bands] > avg_threshold_db[active_bands]),
                 alpha=0.3, label='Masking Benefit')

plt.title('Bark Band Energy vs. Psychoacoustic Masking Threshold', fontsize=14)
plt.xlabel('Bark Band Index')
plt.ylabel('Energy (dB, relative)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3. MDCT coefficient comparison
frame_idx = T_frames // 2  # Middle frame

plt.figure(figsize=(14, 8))
plt.semilogy(freqs, np.abs(X_frames[frame_idx]) + 1e-12, 'b-', 
             label='Original MDCT', linewidth=1.5)
plt.semilogy(freqs, np.abs(Xq_frames[frame_idx]) + 1e-12, 'r--', 
             label='Quantized MDCT', linewidth=1.5, alpha=0.8)

plt.title(f'MDCT Magnitude Spectrum (Frame {frame_idx})', fontsize=14)
plt.xlabel('Frequency (Hz)')
plt.ylabel('|MDCT Coefficient|')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, fs/2)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Absolute Threshold of Hearing visualization
freq_range = np.logspace(1.3, 4.3, 1000)  # 20 Hz to 20 kHz
ath_curve = absolute_threshold_of_hearing(freq_range)

plt.figure(figsize=(12, 6))
plt.semilogx(freq_range, ath_curve, 'g-', linewidth=2, label='Absolute Threshold of Hearing')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5, label='0 dB Reference')

# Mark the frequencies in our test signal
test_freqs = [440, 1000, 3500]
for f in test_freqs:
    ath_val = absolute_threshold_of_hearing(np.array([f]))[0]
    plt.axvline(x=f, color='r', linestyle=':', alpha=0.7)
    plt.text(f, ath_val+5, f'{f} Hz', rotation=90, ha='center', va='bottom')

plt.title('Absolute Threshold of Hearing', fontsize=14)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Threshold Level (dB SPL)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.xlim(20, 20000)
plt.ylim(-10, 80)
plt.tight_layout()
plt.show()

## 11. Audio Export for Listening Tests

Save the original and compressed audio for subjective evaluation.

In [ ]:
# Normalize both signals using the same scale factor
max_val = np.max(np.abs(x_comp)) + 1e-12
x_out = (x_comp / max_val * 0.95).astype(np.float32)
y_out = (y_comp / max_val * 0.95).astype(np.float32)

# Save as WAV files
try:
    wavfile.write('original_audio.wav', fs, x_out)
    wavfile.write('compressed_audio.wav', fs, y_out)
    print("Audio files saved:")
    print("  - original_audio.wav")
    print("  - compressed_audio.wav")
    print("\nListen to both files to evaluate the perceptual quality.")
    print("The compressed version should sound very similar to the original")
    print("despite the quantization, thanks to psychoacoustic masking.")
except Exception as e:
    print(f"Could not save audio files: {e}")
    print("Audio data is available in variables x_out and y_out")

## 12. Summary and Key Insights

This implementation demonstrates the core principles of modern perceptual audio compression:

### Key Results:
- **Perfect MDCT reconstruction** when quantization is disabled
- **Psychoacoustic masking** reduces audible artifacts from quantization
- **Bark-scale analysis** groups frequencies according to auditory perception
- **Adaptive quantization** shapes noise below the masking threshold

### Technical Achievements:
1. **Corrected MDCT implementation** with proper transform kernel
2. **Bark-scale frequency mapping** using Traunmüller approximation
3. **Psychoacoustic model** combining ATH and frequency spreading
4. **Perceptual quantization** with proper noise power calculation

### Applications:
This approach forms the foundation for:
- **MP3** (MPEG-1 Layer III)
- **AAC** (Advanced Audio Coding)
- **Vorbis** (Ogg Vorbis)
- **Opus** (modern low-latency codec)

### Further Reading:
For deeper understanding of audio compression and psychoacoustics:

1. **Spanias, A., Painter, T., & Atti, V. (2006)**. *Audio signal processing and coding*. John Wiley & Sons.

2. **Bosi, M., & Goldberg, R. E. (2012)**. *Introduction to digital audio coding and standards*. Springer Science & Business Media.

3. **Moore, B. C. (2012)**. *An introduction to the psychology of hearing*. Brill.

4. **Watkinson, J. (2001)**. *The art of digital audio*. Focal Press.

### Conclusion:
Perceptual audio compression achieves high compression ratios by exploiting the limitations and masking properties of human hearing. The combination of time-frequency analysis (MDCT), perceptual modeling (Bark scale + psychoacoustics), and adaptive quantization enables transparent compression at moderate bitrates.

The key insight is that **not all quantization noise is equally audible** - by shaping the noise spectrum to stay below the masking threshold, we can achieve significant compression while maintaining perceptual quality.